# 07.08 - GAN latent-space exploration and evaluation

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** Latent interpolation and compact GAN evaluation report.

This practice day separates generator inspection from training. You will make controlled latent paths and calculate small, interpretable diversity and fidelity proxies.

## Core Ideas

Fixed latent vectors make comparisons across checkpoints fair. Interpolation tests whether nearby latent codes produce smooth output changes. Diversity and nearest-reference distance measure different properties: a generator can be diverse but unrealistic, or realistic-looking but collapsed. Tiny proxies are diagnostics, not substitutes for FID or human review.

In [ ]:
import numpy as np
import torch
from torch import nn

SEED = 7
np.random.seed(SEED)
torch.manual_seed(SEED)

## Prepared Generator Boundary

A deterministic linear layer stands in for an already-trained tiny generator. The lesson focuses on evaluation mechanics, so no checkpoint download or training is required.

**Model return structure — `toy_generator`:** For a CPU float tensor `[N,3]`, returns a CPU float tensor `[N,2]`.

In [ ]:
toy_generator = nn.Linear(3, 2, bias=True)
with torch.no_grad():
    toy_generator.weight.copy_(torch.tensor([[0.9, -0.2, 0.4], [0.1, 0.8, -0.5]]))
    toy_generator.bias.copy_(torch.tensor([0.05, -0.05]))

reference_angles = torch.linspace(0.0, 2.0 * float(np.pi), 33)[:-1]
real_reference = torch.stack([torch.cos(reference_angles), torch.sin(reference_angles)], dim=1)
z_start = torch.tensor([-1.0, 0.0, 0.5])
z_end = torch.tensor([1.0, 0.5, -0.5])
print("generator/reference:", toy_generator, real_reference.shape)

## Exercise 07-A: Interpolate latent codes

Include both endpoints and use one shared interpolation coefficient for every latent dimension.

**Return structure — `interpolate_latents`:** A CPU `torch.float32` tensor of shape `[steps,D]`, where `D` is the latent dimension. Row 0 equals `start`; row `steps-1` equals `end`.

In [ ]:
# TODO 07-A
def interpolate_latents(start, end, steps=9):
    raise NotImplementedError("Complete Exercise 07-A")


# Smoke check: create a nine-point path.
latent_path = interpolate_latents(z_start, z_end, steps=9)
print("latent path:", latent_path.shape, latent_path[0], latent_path[-1])

## Exercise 07-B: Generate the complete path

Evaluation must use `eval()` and `torch.no_grad()` so BatchNorm/dropout behavior and autograd memory are controlled.

**Return structure — `generate_latent_path`:** A detached CPU `torch.float32` tensor with shape `[N,O]`, where `N` is the number of latent rows and `O` is the generator output dimension.

In [ ]:
# TODO 07-B
def generate_latent_path(generator, latent_codes):
    raise NotImplementedError("Complete Exercise 07-B")


# Smoke check: generate every interpolation point in one batch.
generated_path = generate_latent_path(toy_generator, latent_path)
print("generated path:", generated_path.shape)

## Exercise 07-C: Measure diversity and fidelity proxies

Mean pairwise distance is a collapse signal; mean distance to the closest real reference point is a compact fidelity proxy.

**Return structure — `compact_gan_metrics`:** A dictionary with Python floats `diversity` and `nearest_reference_distance`, integer `sample_count`, and boolean `possible_collapse`.

In [ ]:
# TODO 07-C
def compact_gan_metrics(generated, reference, collapse_threshold=0.05):
    raise NotImplementedError("Complete Exercise 07-C")


# Smoke check: score the generated interpolation path.
path_metrics = compact_gan_metrics(generated_path, real_reference)
print("path metrics:", path_metrics)

## Exercise 07-D: Compare a collapsed sample set

Apply the same metric function to a deliberately repeated output and write a compact comparison record.

**Return structure — `compare_generator_samples`:** A `list[dict]` with one row per named sample set. Each row has `name` (`str`) plus all four keys returned by `compact_gan_metrics`.

In [ ]:
# TODO 07-D
def compare_generator_samples(named_samples, reference):
    raise NotImplementedError("Complete Exercise 07-D")


# Smoke check and full evidence: compare the path with repeated samples.
collapsed_samples = generated_path[:1].repeat(len(generated_path), 1)
gan_comparison = compare_generator_samples({"interpolation": generated_path, "collapsed": collapsed_samples}, real_reference)
print("comparison:", gan_comparison)

## Test Cases

**Return structure — `run_day07_tests`:** Returns `None`; assertions and `Day 07 tests passed` communicate success.

In [ ]:
def run_day07_tests():
    assert latent_path.shape == (9, 3) and latent_path.dtype == torch.float32
    assert torch.allclose(latent_path[0], z_start) and torch.allclose(latent_path[-1], z_end)
    assert generated_path.shape == (9, 2) and not generated_path.requires_grad
    assert set(path_metrics) == {"diversity", "nearest_reference_distance", "sample_count", "possible_collapse"}
    rows = {row["name"]: row for row in gan_comparison}
    assert set(rows) == {"interpolation", "collapsed"}
    assert rows["collapsed"]["diversity"] == 0.0 and rows["collapsed"]["possible_collapse"]
    assert rows["interpolation"]["diversity"] > rows["collapsed"]["diversity"]
    print("Day 07 tests passed")


run_day07_tests()

## Day 07 Checklist

- [ ] Use fixed latent endpoints for fair comparisons.
- [ ] Generate paths in evaluation mode without gradients.
- [ ] Separate diversity from fidelity evidence.
- [ ] Recognize a deliberately collapsed sample set.
- [ ] Run the test cases.